# Vintage CORE Evaluation

Run one or all three base models on full **Original CORE**, **Filtered CORE**, and **Restyled CORE** from the `Vintage-CORE` branch of `zachnorton14/think.nano`. Models run sequentially in isolated subprocesses and write to separate result directories. Original CORE has 22 tasks; the Vintage bundles have 20, so the notebook reports both each bundle's native aggregate and a directly comparable Common-20 aggregate. LAMBADA is informational and never triggers fallback or blocks a result.

## 1. Configuration
Set `RUN_ALL_MODELS=True` for the complete comparison. GPT-1900 d34 should use an A100-class Colab runtime.

In [ ]:
MODEL_ID = "think-d12-r30"
RUN_ALL_MODELS = True
RUN_FULL = True
LOG_TO_WANDB = False

VALID_MODELS = ["think-d12-r30", "modern-d24", "gpt1900-d34"]
assert MODEL_ID in VALID_MODELS
assert RUN_FULL is True, "The publication runs are full evaluations."
MODELS_TO_RUN = VALID_MODELS if RUN_ALL_MODELS else [MODEL_ID]
print("Models to run:", ", ".join(MODELS_TO_RUN))

## 2. Environment
Select **Runtime → Change runtime type → GPU** first. While the dataset is private, add an `HF_TOKEN` secret in Colab (key icon at left).

In [ ]:
import os, platform, subprocess, sys
import torch

assert torch.cuda.is_available(), "A CUDA GPU is required."
props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name}")
print(f"VRAM: {props.total_memory / 2**30:.1f} GiB")
print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
if "gpt1900-d34" in MODELS_TO_RUN and "A100" not in props.name:
    print("WARNING: GPT-1900 d34 is intended for an A100-class runtime.")

REPO_URL = "https://github.com/zachnorton14/think.nano.git"
REPO_BRANCH = "Vintage-CORE"
repo = "/content/think.nano"
if not os.path.exists(repo):
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, repo], check=True)
else:
    subprocess.run(["git", "-C", repo, "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", repo, "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", repo, "pull", "--ff-only", "origin", REPO_BRANCH], check=True)
branch = subprocess.check_output(["git", "-C", repo, "branch", "--show-current"], text=True).strip()
assert branch == REPO_BRANCH, f"Expected {REPO_BRANCH}, found {branch}"
print(f"Repository: {REPO_URL} @ {branch}")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{repo}/dev/vintage_core_colab/requirements.txt"], check=True)

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
except Exception:
    pass
print("HF_TOKEN available:", bool(os.environ.get("HF_TOKEN")))

## 3. Download, validate, load, and evaluate
For each selected model, the evaluator downloads only its checkpoint, metadata, tokenizer, and pinned runtime. It validates every bundle file, runs one forward-pass loader check, then evaluates the three full bundles sequentially. Each model has its own output directory and a JSON result is saved after every bundle. A failed model is reported without deleting completed results from other models.

In [ ]:
RESULTS_ROOT = "/content/vintage-core-results"
def stream_command(command):
    environment = os.environ.copy()
    environment["PYTHONUNBUFFERED"] = "1"
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=environment,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
    return process.wait()

RUN_STATUS = {}
for model_id in MODELS_TO_RUN:
    output_dir = f"{RESULTS_ROOT}/{model_id}"
    if os.path.isfile(f"{output_dir}/summary.csv"):
        print(f"ALREADY COMPLETE: {model_id}")
        RUN_STATUS[model_id] = 0
        continue
    command = [
        sys.executable, "-u", f"{repo}/dev/vintage_core_colab/vintage_core_eval.py",
        "--model", model_id,
        "--bundles", "original,filtered,restyled",
        "--output-dir", output_dir,
        "--max-per-task", "-1",
    ]
    print("\nRunning:", " ".join(command), flush=True)
    returncode = stream_command(command)
    RUN_STATUS[model_id] = returncode
    if returncode:
        print(f"FAILED: {model_id} (exit {returncode}). Scroll up to the first traceback for the cause.")
    else:
        print(f"COMPLETED: {model_id}")
print("Run status:", RUN_STATUS)

## 4. Results

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

summaries = []
COMPLETED_MODELS = []
for model_id in MODELS_TO_RUN:
    summary_path = Path(RESULTS_ROOT) / model_id / "summary.csv"
    if summary_path.is_file():
        frame = pd.read_csv(summary_path)
        frame["model"] = model_id
        summaries.append(frame)
        COMPLETED_MODELS.append(model_id)
    else:
        print(f"Skipping charts for incomplete model: {model_id}")
if not summaries:
    raise FileNotFoundError("No completed summary.csv files. Inspect the evaluator traceback above.")
summary_all = pd.concat(summaries, ignore_index=True)
summary_all.to_csv(f"{RESULTS_ROOT}/combined_summary.csv", index=False)
display(summary_all.style.format({"native_core": "{:.4f}", "common_20_core": "{:.4f}", "runtime_seconds": "{:.1f}"}))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for axis, metric, title in zip(axes, ["native_core", "common_20_core"], ["Native CORE", "Common-20 CORE"]):
    summary_all.pivot(index="model", columns="bundle", values=metric).plot(kind="bar", ax=axis)
    axis.set_title(title)
    axis.set_xlabel("")
    axis.set_ylabel("Centered CORE")
    axis.tick_params(axis="x", rotation=20)
    axis.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
accuracy_frames, delta_frames = [], []
for model_id in COMPLETED_MODELS:
    model_dir = Path(RESULTS_ROOT) / model_id
    accuracy = pd.read_csv(model_dir / "task_accuracy.csv")
    accuracy["model"] = model_id
    accuracy_frames.append(accuracy)
    deltas = pd.read_csv(model_dir / "task_deltas.csv")
    deltas["model"] = model_id
    delta_frames.append(deltas)
accuracy_all = pd.concat(accuracy_frames, ignore_index=True)
deltas_all = pd.concat(delta_frames, ignore_index=True)
accuracy_all.to_csv(f"{RESULTS_ROOT}/combined_task_accuracy.csv", index=False)
deltas_all.to_csv(f"{RESULTS_ROOT}/combined_task_deltas.csv", index=False)

lambada = accuracy_all[accuracy_all["task"] == "lambada_openai"].set_index("model")[["original", "filtered", "restyled"]]
axis = lambada.plot(kind="bar", figsize=(9, 5), title="LAMBADA raw accuracy")
axis.set_xlabel("")
axis.set_ylabel("Raw accuracy")
axis.tick_params(axis="x", rotation=20)
axis.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

print("LAMBADA comparison")
display(lambada.style.format("{:.4f}"))
print("All raw per-task accuracies")
display(accuracy_all.set_index(["model", "task"]).style.format("{:.4f}"))
print("All bundle deltas")
display(deltas_all.set_index(["model", "task"]).style.format("{:+.4f}"))

## 5. Optional W&B logging and download
W&B is not required. The local JSON and CSV files are the canonical notebook outputs.

In [ ]:
if LOG_TO_WANDB:
    import wandb
    run = wandb.init(project="vintage-core", name="all-models-core-matrix")
    wandb.log({"core_comparison": wandb.Table(dataframe=summary_all)})
    run.finish()

# Uncomment in Colab to download all small result files.
# from google.colab import files
# archive = __import__("shutil").make_archive("/content/vintage-core-results", "zip", RESULTS_ROOT)
# files.download(archive)